In [34]:
import duckdb 

con = duckdb.connect()

con.sql("CREATE OR REPLACE VIEW orders AS SELECT * FROM '../data/raw/olist_orders_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW order_items AS SELECT * FROM '../data/raw/olist_order_items_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW customers AS SELECT * FROM '../data/raw/olist_customers_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW products AS SELECT * FROM '../data/raw/olist_products_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW translation AS SELECT * FROM '../data/raw/product_category_name_translation.csv'")
con.sql("CREATE OR REPLACE VIEW order_reviews AS SELECT * FROM '../data/raw/olist_order_reviews_dataset.csv'")
con.sql("SHOW TABLES").show()

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ customers     │
│ order_items   │
│ order_reviews │
│ orders        │
│ products      │
│ translation   │
└───────────────┘



In [26]:
#1. 
con.sql("""
    SELECT COUNT(*) AS total_delivered_orders
    FROM orders
    WHERE order_status = 'delivered'
""").show()


┌────────────────────────┐
│ total_delivered_orders │
│         int64          │
├────────────────────────┤
│                  96478 │
└────────────────────────┘



In [27]:
#2. 
con.sql("""
    SELECT 
        customer_state,
        COUNT(*) AS total_clientes
    FROM customers
    GROUP BY customer_state
    ORDER BY total_clientes DESC
    LIMIT 5
""")

customer_stateVARCHAR,total_clientesBIGINT
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045


In [28]:
#3. 
con.sql("""
    SELECT 
        ROUND(SUM(oi.price), 2) AS total_revenue,
        ROUND(AVG(oi.price), 2) AS avg_item_price
    FROM orders o
    INNER JOIN order_items AS oi
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
""")

total_revenueDOUBLE,avg_item_priceDOUBLE
13221498.11,119.98


In [29]:
#4. 
con.sql("""
    SELECT 
        oi.seller_id,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        COUNT(*) AS items_sold
    FROM orders o
    INNER JOIN order_items oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
    HAVING COUNT(*) > 1000
    ORDER BY total_revenue DESC;
""")

seller_idVARCHAR,total_revenueDOUBLE,items_soldBIGINT
4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,1148
4a3ca9315b744ce9f8e9374361493884,196882.12,1949
7c67e1448b00f6e969d365cea6b010ab,186570.05,1355
da8622b14eb17ae2831f4ac5b9dab84a,159816.87,1548
7a67c85e85bb2ce8582c35f2203ad736,139658.69,1155
1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,1420
955fee9216a65b617aa5c0531780ce60,131836.71,1472
6560211a19b47992c3666cc44a7e94c0,120702.83,1996
1f50f920176fa81dab994f9023523100,106655.71,1926
cc419e0650a3c5ba77189a1882b7556a,101090.96,1719


In [30]:
#5. 
con.sql("""
    SELECT 
        t.product_category_name_english AS category,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        ROUND(COUNT(*), 2) AS items_saled,
        ROUND(AVG(oi.price), 2) AS avg_items
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    JOIN translation t
        ON p.product_category_name = t.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY t.product_category_name_english
    ORDER BY total_revenue DESC
    LIMIT 5
""")

categoryVARCHAR,total_revenueDOUBLE,items_saledBIGINT,avg_itemsDOUBLE
health_beauty,1233131.72,9465,130.28
watches_gifts,1166176.98,5859,199.04
bed_bath_table,1023434.76,10953,93.44
sports_leisure,954852.55,8431,113.25
computers_accessories,888724.61,7644,116.26


In [31]:
#6. 
con.sql("""
    WITH order_item_counts AS (
        SELECT 
            o.order_id,
            COUNT(oi.order_item_id) AS items_count
        FROM orders o
        JOIN order_items oi 
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY o.order_id
    )
    SELECT 
        CASE 
            WHEN items_count = 1 THEN 'Single Item'
            WHEN items_count BETWEEN 2 AND 3 THEN 'Small Basket'
            ELSE 'Large Basket'
        END AS basket_segment,
        COUNT(*) AS total_orders,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_orders
    FROM order_item_counts
    GROUP BY basket_segment
    ORDER BY total_orders DESC; 
""")

basket_segmentVARCHAR,total_ordersBIGINT,pct_ordersDOUBLE
Single Item,86843,90.01
Small Basket,8698,9.02
Large Basket,937,0.97


In [32]:
#7
con.sql("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            SUM(oi.price) AS order_value
        FROM orders o
        JOIN order_items oi
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY o.order_id
    )
    SELECT
        CASE
            WHEN order_value < 50 THEN 'Low Ticket'
            WHEN order_value BETWEEN 50 AND 200 THEN 'Medium Ticket'
            ELSE 'High Ticket'
        END AS ticket_tier,
        COUNT(*) AS total_orders,
        ROUND(SUM(order_value), 2) AS total_revenue,
        ROUND(SUM(order_value) * 100 / SUM(SUM(order_value)) OVER (), 2) AS pct_revenue
    FROM order_totals
    GROUP BY ticket_tier
    ORDER BY total_revenue DESC;
""")


ticket_tierVARCHAR,total_ordersBIGINT,total_revenueDOUBLE,pct_revenueDOUBLE
High Ticket,14538,6554012.07,49.57
Medium Ticket,53149,5750894.33,43.5
Low Ticket,28791,916591.71,6.93


In [35]:
con.sql("""
    WITH order_region_data AS (
        SELECT 
            o.order_id,
            CASE
                WHEN c.customer_state = 'SP' THEN 'Sao Paulo'
                WHEN c.customer_state IN ('RJ', 'MG', 'ES') THEN 'Sudeste'
                WHEN c.customer_state IN ('PR', 'SC', 'RS') THEN 'Sur'
                ELSE 'Otras Regiones'
            END AS macro_regiones,
            r.review_score,
            CASE
                WHEN o.order_delivered_customer_date > order_estimated_delivery_date THEN 1
                ELSE 0
            END AS is_late
        FROM orders o
        JOIN customers c 
            ON o.customer_id = c.customer_id
        JOIN order_reviews r
            ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT 
        macro_regiones,
        COUNT(*) AS total_orders,
        ROUND(AVG(review_score), 2) AS avg_review_score,
        ROUND(AVG(is_late) * 100, 2) AS pct_late_orders
    FROM order_region_data
    GROUP BY macro_regiones
    HAVING COUNT(*) > 5000
    ORDER BY macro_regiones ASC;
""")

macro_regionesVARCHAR,total_ordersBIGINT,avg_review_scoreDOUBLE,pct_late_ordersDOUBLE
Otras Regiones,16448,4.03,11.47
Sao Paulo,40477,4.25,5.82
Sudeste,25613,4.07,9.73
Sur,13815,4.19,6.99
